# Low input proteomics analysis

In [ ]:
from low_input import *
#%load_ext autoreload
#%autoreload 2

id_columns = ["uniprot", "protein", "description"]
reactivity_id_columns = ["identifier", "uniprot", "sequence", "residue", "description", "protein"]

# applied to remove hemoglobin proteins from in vivo data
hemoglobin_check = pl.col("description").str.contains("Hemoglobin")

# Inhibitor whole proteome data

In [ ]:
inhibitor_palette = {
    "D2": "#A1A1A1",
    "D8A + DMSO": "#603EA6",
    "D8C + DMSO": "#E27753",
    "D8C + Gossypetin": "#A3425B",
    "D8A + Thapsigargin": "#8FB59D",
    "D8C + Thapsigargin": "#3D7A54",
    "D8C + ISRIB": "#C3BB20",
    "D8C + BSJ": "#4287A3",
}

donor_sets = {
    "both_4_donors":{"HK3_60": "d1", "HK3_62": "d2", "HK3_73": "d3", "HK3_74": "d4"},
    "vardhana_2_donors":{"HK3_60": "d1", "HK3_62": "d2"},
    "vinogradova_2_donors":{"HK3_73": "d3", "HK3_74": "d4"},
}

data_dir = Path("/Users/henrysanford/dev/test_data/inhibitor_treatment/both_4_donors")


inhibitor_donors = {"HK3_60": "d1", "HK3_62": "d2", "HK3_73": "d3", "HK3_74": "d4"}

wp_channel_ratio_df = read_channel_ratios(
    data_dir=Path("/Users/henrysanford/dev/test_data/inhibitor_treatment/"),
    donors=inhibitor_donors,
    curated_filter_expression=curated_filter_expression,
    folder_suffix="_WP_one peptide/03_combined_files",
)

df_normalized = median_normalize_data(
    df=wp_channel_ratio_df, height=10, color_palette=inhibitor_palette
)

### PCA

In [ ]:
pca_dir = data_dir / "pca"
os.makedirs(pca_dir, exist_ok=True)
pca_input_df = (
    df_normalized.drop("peptide_number")
    .filter(pl.col("condition").eq("D8C + Gossypetin").not_())
    .pivot(index=id_columns, values="value", on="clean_channel_name")
    .drop(["uniprot", "description"])
)

pca_plt, loadings_df, pca_df, percent_explained_df = get_pca_plot(
    df=pca_input_df.to_pandas(),
    index_cols="protein",
    title_string="Whole proteome",
    out_dir=Path(pca_dir),
)
pca_plt.show()

In [ ]:
bargraph_dir = Path(data_dir  / "bargraphs")
bargraph_dir.mkdir(exist_ok=True)

percent_controls = channel_ratio_to_percent_control(
    df_normalized,
    "D2",
    id_columns + ["donor", "peptide_number"]
)



percent_controls.write_csv(
    bargraph_dir / "long_percent_control.csv"
)

### GSEA

In [ ]:
gsea_dir = Path(data_dir  / "gsea")
gsea_dir.mkdir(exist_ok=True)

comparisons = [
    ["D8C + DMSO", "D8C + BSJ"],
    ["D8C + DMSO", "D8C + Gossypetin"],
    ["D8C + DMSO", "D8C + ISRIB"],
    ["D8C + DMSO", "D8C + Thapsigargin"],
    ["D8A + DMSO", "D8A + Thapsigargin"],
    ["D2", "D8C + DMSO"],
    ["D2", "D8A + DMSO"],
    
]

results = []

for comparison in comparisons:

    gsea_input = (
        wp_channel_ratio_df.filter(pl.col("condition").is_in(comparison))
        .group_by(id_columns + ["condition"])
        .agg(pl.col("value").mean())
        .pivot(index=id_columns, on="condition")
        .with_columns(
            (pl.col(comparison[1]) / pl.col(comparison[0])).log(base=2).alias("LFC")
        )
        .sort(by="LFC")
        .select(["protein", "LFC"])
        .to_pandas()
        .set_index("protein")
    )

    # Perform GSEA
    gsea_results = gp.prerank(
        rnk=gsea_input,
        gene_sets="t_cell_dysfunction.gmt",
        processes=4,
        correction="bh",
        permutation_num=50000,
        outdir="./",
        seed=42,
        no_plot=True,
    )

    result = pl.from_pandas(gsea_results.res2d).with_columns(
        pl.lit(comparison[0]).alias("control_condition"),
        pl.lit(comparison[1]).alias("treatment_condition"),
    )

    results.append(result)
all_gsea_results = pl.concat(results, how = "vertical")

all_gsea_results.write_csv(gsea_dir / "gsea_results.csv")

In [ ]:
type_filter = "standard"
if CURATED_FILTERS:
    type_filter = "curated"
fn = data_dir / f"inhibitor_wp_exhaustion_{type_filter}_filters.xlsx"
write_percent_control_to_excel(
    df = percent_controls,
    file_name = fn
)

# Inhibitor reactivity

In [ ]:
dataset_version = "both_4_donors"
data_dir = Path(f"/Users/henrysanford/dev/test_data/inhibitor_reactivity/{dataset_version}")
data_dir.mkdir(exist_ok=True)



wp_channel_ratio_df = read_channel_ratios(
    data_dir=Path("/Users/henrysanford/dev/test_data/inhibitor_treatment/"),
    donors=inhibitor_donors,
    curated_filter_expression=curated_filter_expression,
    folder_suffix="_WP_one peptide/03_combined_files",
)

In [ ]:
df = (
    pl.read_csv(
        "/Users/henrysanford/dev/test_data/inhibitor_reactivity/combine_runs/04_combined_files/2/05_combfiles_forpca_cysaggr_channelratio_rc.csv"
    )
    .drop("")
    .unpivot(index=reactivity_id_columns)
    .filter(pl.col("uniprot") != "contaminant")
    .with_columns(
        # rename conditions
        pl.col("variable").str.replace_many(
            ORIGINAL_LABELS,
            CLEANED_LABELS,
        ),
    )
    .with_columns(
        pl.col("variable")
        .str.split(pl.lit("_processed_census-out_"))
        .list.get(0)
        .alias("condition,tech_rep"),
        pl.col("variable")
        .str.split(pl.lit("_processed_census-out_"))
        .list.get(1)
        .alias("donor"),
    )
    .with_columns(
        pl.col("condition,tech_rep").str.tail(1).alias("technical_replicate"),
        pl.col("condition,tech_rep")
        .str.head(-2)
        .alias("condition")
        .str.replace_many(ORIGINAL_LABELS, CLEANED_LABELS),
        pl.col("donor").replace(
            old=[
                "HK3-60_Reactivity",
                "20251130_HK3-62_Reactivity",
                "20251230_HK3-73_Reactivity",
                "20251231_HK3-74_Reactivity",
            ],
            new=["d1", "d2", "d3", "d4"],
        ),
    )
    .with_columns(
        pl.concat_str(
            [pl.col("condition"), pl.col("donor"), pl.col("technical_replicate")]
        ).alias("clean_channel_name")
    )
    .drop(["condition,tech_rep"])
    .drop_nulls(subset="value")
    .filter(pl.col("donor").is_in(donor_sets[dataset_version].values()))
)


curated_filter_expression = (pl.col("protein").is_in(TRUE_POSITIVE_LIST)) & CURATED_FILTERS

channel_ratio_df = filter_to_two_reps(
    curated_filter_expression,
    id_columns=reactivity_id_columns,
    allow_one_rep=False,
    channel_ratio_df=df,
).filter(
    pl.col("condition") != "D8C + Gossypetin"
)

In [ ]:



channel_ratio_df = median_normalize_data(
    channel_ratio_df, 10, color_palette=inhibitor_palette
)

### Non-unique peptide

Channel ratio and median normalization for a single peptide that was filtered out of the main dataset for belonging to two proteins

In [ ]:
peptide_sequence = "K.MC*DFGISGYLVDSVAK.T"

census_out_dir = Path(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Exhaustion manuscript/02_Mass-spectrometry data/12_low-input proteomics with inhibitors/census-out-processing"
)

# read all the non-unique peptide files
peptide_signal_intensities = []
for name, abbrev in inhibitor_donors.items():
    name = name.replace("_", "-") + "_Reactivity"
    df = (
        pl.read_csv(
            census_out_dir
            / (
                name
                + "/02_filteredandusedpeptides_files/census-out_"
                + name
                + "/not_unique_peptides.csv"
            ),
            ignore_errors=True,
        )
        .filter(pl.col("SEQUENCE") == peptide_sequence)
        .with_columns(pl.lit(abbrev).alias("donor"))
    )

    df = df.drop([df.columns[0], df.columns[1], df.columns[2]])

    peptide_signal_intensities.append(df)

census_out_id_cols = ["SEQUENCE", "donor"]

peptide_data = (
    pl.concat(peptide_signal_intensities, how="vertical_relaxed")
    # unique rows because census out report the peptide mulitple times
    # for each protein it's in
    .unique()
    # average within each donor because peptides are in multiple fractions
    .with_columns(cs.contains("tag").cast(pl.Float64))
    .group_by(census_out_id_cols)
    .agg(cs.contains("tag").mean())
    .unpivot(on=cs.contains("tag"), index=~cs.contains("tag"))
)


# normalize SI to channel ratio
peptide_data = (
    peptide_data.group_by(census_out_id_cols)
    .agg(pl.col("value").sum().alias("sum"))
    .join(other=peptide_data, on=census_out_id_cols)
    .with_columns(pl.col("value").truediv(pl.col("sum")).alias("channel_ratio"))
)


# apply median normalization to channel ratio
normalization_factors = channel_ratio_df.select(
    ["clean_channel_name", "normalization_factor"]
).unique()
channel_metadata = {
    "tag_126.127726": "D2_x_1",
    "tag_127.124761": "D2_x_2",
    "tag_127.131081": "D8A + DMSO_x_1",
    "tag_128.128116": "D8A + DMSO_x_2",
    "tag_128.134436": "D8C + DMSO_x_1",
    "tag_129.131471": "D8C + DMSO_x_2",
    "tag_129.13779": "D8C + BSJ_x_1",
    "tag_130.134825": "D8C + BSJ_x_2",
    "tag_130.141145": "D8C + Gossypetin_x_1",
    "tag_131.13818": "D8C + Gossypetin_x_2",
    "tag_131.1445": "D8C + ISRIB_x_1",
    "tag_132.141535": "D8C + ISRIB_x_2",
    "tag_132.147855": "D8A + Thapsigargin_x_1",
    "tag_133.14489": "D8A + Thapsigargin_x_2",
    "tag_133.15121": "D8C + Thapsigargin_x_1",
    "tag_134.148245": "D8C + Thapsigargin_x_2",
}
peptide_data = (
    peptide_data.with_columns(
        pl.col("variable")
        .replace(channel_metadata)
        .str.replace("x", pl.col("donor"))
        .alias("clean_channel_name")
    )
    .join(other=normalization_factors, on="clean_channel_name")
    .drop("value")
    .with_columns(
        pl.col("channel_ratio").mul("normalization_factor").alias("value"),
        pl.col("clean_channel_name").str.split("_").list.get(0).alias("condition"),
    )
    .drop(["variable", "sum", "normalization_factor", "channel_ratio"])
)

channel_ratio_to_percent_control(
    peptide_data, control_condition="D2", grouping_columns=["SEQUENCE", "donor"]
).drop("D2_median").write_csv(
    data_dir / (peptide_sequence.replace(".", "").replace("*", "") + ".csv")
)

## comparison to wp

In [ ]:
bargraph_dir = Path(data_dir / "processed_results" / "bargraphs")
bargraph_dir.mkdir(exist_ok=True, parents=True)
channel_ratio_df = channel_ratio_df.filter(pl.col("uniprot") != "contaminant")
percent_to_lfc = pl.col("percent_control").truediv(100).log(base=2)
add_residue_loc = (
    pl.col("residue").str.replace_all(";", ","),
    pl.col("residue")
    .str.split(";")
    .list.get(0)
    .str.slice(offset=1)
    .cast(pl.Int64)
    .alias("residue_loc"),
)
results = []
for control_condition in ["D2", "D8A + DMSO", "D8C + DMSO"]:
    # first format the reactivity data
    percent_controls = channel_ratio_to_percent_control(
        channel_ratio_df,
        control_condition=control_condition,
        grouping_columns=[
            "identifier",
            "uniprot",
            "sequence",
            "residue",
            "description",
            "protein",
        ]
        + ["donor"],
    ).with_columns(add_residue_loc)

    # format the wp data
    # it needs to be normalized to the same condition as the
    # reactivity data
    averaged_wp = (
        channel_ratio_to_percent_control(
            wp_channel_ratio_df,
            control_condition=control_condition,
            grouping_columns=id_columns + ["donor", "peptide_number"],
        )
        .select(id_columns + ["percent_control", "condition"])
        .group_by(id_columns + ["condition"])
        .agg(pl.col("percent_control").median())
        .with_columns(percent_to_lfc.alias("LFC_WP"))
        .rename({"percent_control": "wp_percent_control"})
    )

    rc_id_columns = id_columns + ["sequence", "residue", "residue_loc"]

    averaged_rc = (
        percent_controls.select(rc_id_columns + ["percent_control", "condition"])
        .group_by(rc_id_columns + ["condition"])
        .agg(pl.col("percent_control").median())
        .with_columns(percent_to_lfc.alias("LFC_RC"))
        .rename({"percent_control": "rc_percent_control"})
    )

    rc_protein_medians = averaged_rc.group_by(id_columns + ["condition"]).agg(
        pl.col("rc_percent_control")
        .median()
        .alias("median_percent_control_of_cysteines")
    )

    df = (
        averaged_rc.join(rc_protein_medians, on=id_columns + ["condition"], how="left")
        .join(other=averaged_wp, on=id_columns + ["condition"], how="left")
        .with_columns(
            pl.when(pl.col("wp_percent_control").is_null())
            .then(
                pl.col("rc_percent_control")
                / pl.col("median_percent_control_of_cysteines")
            )
            .otherwise(pl.col("rc_percent_control") / pl.col("wp_percent_control"))
            .alias("ratio_to_expression")
        )
        .with_columns(
            (
                (pl.col("ratio_to_expression") > 2)
                | (pl.col("ratio_to_expression") < 0.5)
            ).alias("reactivity_change"),
            pl.lit(control_condition).alias("control_condition"),
        )
    )

    results.append(df)

combined_results = pl.concat(results, how="vertical")

combined_results.write_csv(data_dir / "rc_df_long_format.csv")

In [ ]:
percent_controls = channel_ratio_to_percent_control(
        channel_ratio_df,
        control_condition="D2",
        grouping_columns=[
            "identifier",
            "uniprot",
            "sequence",
            "residue",
            "description",
            "protein",
        ]
        + ["donor"],
    )
type_filter = "standard"
if CURATED_FILTERS:
    type_filter = "curated"

fn = data_dir / f"inhibitor_rc_exhaustion_{type_filter}_filters.xlsx"
write_percent_control_to_excel(
    percent_controls,
    file_name = fn
)

In [ ]:
results = []
for control in ["D2", "D8A + DMSO", "D8C + DMSO"]:
    percent_controls = channel_ratio_to_percent_control(
        channel_ratio_df,
        control_condition=control,
        grouping_columns=[
            "identifier",
            "uniprot",
            "sequence",
            "residue",
            "description",
            "protein",
        ]
        + ["donor"],
    )
    print(percent_controls.columns)
    results.append(
        percent_controls.drop(f"{control}_median").with_columns(
            pl.lit(control).alias("control_condition")
        )
    )
pl.concat(results, how="vertical_relaxed").with_columns(add_residue_loc).write_csv(
    data_dir / "long_percent_control.csv"
)

# Mouse in vivo data

conditions: D15 Clone 13,D8 Armstrong,D8 Clone 13,Mouse D2 CD8,Mouse D8A CD8,Mouse D8C CD8,Splenic T cells,TILs

In [ ]:
data_dir = Path("/Users/henrysanford/dev/test_data/texh_revisions/mouse_wp/output")

wp_channel_ratio_df = read_channel_ratios(
    data_dir=data_dir,
    donors={"HK3-80": "d1"},
    folder_suffix="/03_combined_files",
    curated_filter_expression=(pl.col("protein").is_in(MOUSE_TRUE_POSITIVE_LIST))
    & CURATED_FILTERS,
    allow_one_rep=True,
    field_order = ["condition", "drop_2", "technical_replicate", "drop"]
)

# Rename conditions because the pipeline deletes spaces in channel names
old_mouse_conditions = [
    "D15Clone13",
    "D8Armstrong",
    "D8Clone13",
    "MouseD2CD8",
    "MouseD8ACD8",
    "MouseD8CCD8",
    "SplenicTcells",
    "TILs",
]

new_mouse_conditions = [
    "D15 Clone 13",
    "D8 Armstrong",
    "D8 Clone 13",
    "Mouse D2 CD8",
    "Mouse D8A CD8",
    "Mouse D8C CD8",
    "Splenic T cells",
    "TILs",
]

wp_channel_ratio_df = wp_channel_ratio_df.with_columns(
    pl.col("condition").str.replace_many(old_mouse_conditions, new_mouse_conditions),
    pl.col("clean_channel_name").str.replace_many(
        old_mouse_conditions, new_mouse_conditions
    ),
)

In [ ]:
wp_channel_ratio_df = wp_channel_ratio_df.filter(~hemoglobin_check)

In [ ]:
mouse_palette = {
  "Mouse D2 CD8" : "#A1A1A1",
  "Mouse D8A CD8" : "#603EA6",
  "Mouse D8C CD8" : "#E27753",
  "D8 Armstrong" : "#8b8bbc",
  "D8 Clone 13" : "#f7931e",
  "D15 Clone 13" : "#EF3D3B",
  "TILs" : "#e8a0a7",
  "Splenic T cells" : "#a54d5b"
}

normalization_dir = data_dir / "normalization"
os.makedirs(normalization_dir, exist_ok = True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 5), sharey=True, sharex=True)

channel_ratio_boxplot(wp_channel_ratio_df, ax1, "Before normalization", color_palette=mouse_palette)

channel_medians = wp_channel_ratio_df.group_by(["donor", "clean_channel_name"]).agg(
    pl.col("value").median().alias("channel_median")
)

overall_median = wp_channel_ratio_df.select(pl.col("value").median()).item()

df_normalized = (
    wp_channel_ratio_df
    .join(channel_medians, on=["donor", "clean_channel_name"])
    .with_columns(
        (pl.col("value") * overall_median / pl.col("channel_median")).alias("value")
    )
    .drop("channel_median")
)
channel_ratio_boxplot(df_normalized, ax2, "After median normalization", color_palette=mouse_palette)

plt.tight_layout()

plt.savefig(
    normalization_dir / "median_normalization_boxplot.pdf"
)
plt.show()

In [ ]:

pca_dir = data_dir / "pca"
os.makedirs(pca_dir, exist_ok = True)
pca_input_df = df_normalized.drop("peptide_number").pivot(
    index=id_columns, values="value", on="clean_channel_name"
).drop(["uniprot", "description"])

pca_plt, loadings_df, pca_df, percent_explained_df = get_pca_plot(
    df = pca_input_df.to_pandas(),
    
    index_cols =  "protein",
    title_string="Whole proteome",
    out_dir=Path(pca_dir)
)

pca_plt.show()  

In [ ]:
msig = Msigdb()
# get gene set database
# h.all = Hallmark
# c5.go.bp = GO biological process
# c7.immunesigdb = immune signature database
gmt_name = "c5.go.bp"
gmt = msig.get_gmt(gmt_name)

def perform_gsea_proteomics(loading_results, gene_sets, output_dir, pc):
    """
    Perform GSEA on proteomics loading results.
    """
    # Sort proteins by loading score
    ranked_proteins = loading_results.sort_values(ascending=True, by=pc)

    # Perform GSEA
    gsea_results = gp.prerank(
        rnk=ranked_proteins,
        gene_sets=gene_sets,
        threads =8,
        correction="boneferroni",
        permutation_num=5000,
        outdir=output_dir,
        seed=42,
        no_plot=True,
    )

    return gsea_results

In [ ]:


def do_pca(loadings_df, gmt, pca_dir):
    # GSEA of PC loadings

    for pc in ["PC1", "PC2"]:
        print(pc)
        pc_path = pca_dir / pc
        pc_path.mkdir(exist_ok=True)
        pre_res = perform_gsea_proteomics(
            pd.DataFrame(loadings_df[pc]), gene_sets=gmt, output_dir=pc_path, pc=pc
        )

        df = pre_res.res2d
        df["abs_nes"] = -abs(df["NES"])
        df = df[df["NOM p-val"] < 0.01]

        terms = pre_res.res2d.Term[:10]  # Top 5 enriched terms
        fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(10, 10))

        # Dot plot
        gp.plot.dotplot(
            pre_res.res2d,
            title=f"Top enriched terms for {pc}",
            cmap="viridis",
            size=5,
            top_term=15,
            ax=axes,
            column="FDR q-val",
        )
        plt.tight_layout()
        plt.savefig(
            pc_path / "top_enriched_terms.svg", bbox_inches="tight"
        )
        plt.show()


In [ ]:
# in vivo data only
pca_dir = data_dir / "pca" / "in_vivo"
os.makedirs(pca_dir, exist_ok = True)
pca_input_df = df_normalized.drop("peptide_number").pivot(
    index=id_columns, values="value", on="clean_channel_name"
).drop(["uniprot", "description"]).select(~cs.contains("Mouse")) # drop in vitro data

pca_plt, loadings_df, pca_df, percent_explained_df = get_pca_plot(
    df = pca_input_df.to_pandas(),
    
    index_cols =  "protein",
    title_string="Whole proteome",
    out_dir=Path(pca_dir)
)

pca_plt.show()  

In [ ]:
do_pca( loadings_df, gmt, pca_dir)

### Bargraphs

In [ ]:
bargraph_dir = Path(data_dir  / "bargraphs")
bargraph_dir.mkdir(exist_ok=True)

percent_controls = channel_ratio_to_percent_control(
    df_normalized,
    "Mouse D2 CD8",
    id_columns + ["donor", "peptide_number"]
)



percent_controls.write_csv(
    bargraph_dir / "long_percent_control.csv"
)

Signficance test + formatting

In [ ]:
volcano_dir = data_dir / "volcano_plots"
os.makedirs(volcano_dir, exist_ok=True)

comparisons = {
    "Mouse D2 CD8": [
        "Mouse D8A CD8",
        "Mouse D8C CD8",
        "D8 Armstrong",
        "D8 Clone 13",
        "D15 Clone 13",
        "TILs",
    ],
    "Mouse D8C CD8": ["Mouse D8A CD8"],
    "D8 Armstrong": ["D8 Clone 13", "D15 Clone 13"],
    "TILs":["Splenic T cells"]
}

volcano_input = df_normalized.pivot(
    on = "clean_channel_name",
    values = "value",
    index = ["uniprot","protein","description"]
).to_pandas()
results = []
for control, conditions in comparisons.items():
    volcano_df, copy_df, long_df = get_volcano_plot_treatment_vs_control(
        conditions,
        control,
        volcano_input,
        "Mouse WP",
        volcano_dir,
    )
    results.append(long_df)





long_volcano_df = annotate_functional_categories(
    pl.from_pandas(pd.concat(results))
    .filter(~pl.col("uniprot").str.contains("contaminant"))
    .to_pandas().set_index("uniprot")
)

long_volcano_df.to_csv(data_dir / "long_volcano_df.csv")


In [ ]:
index_cols = [
        "protein",
        "description",
        "mitochondrial",
        "peroxisome",
        "redox_related",
        "cell_cycle",
        "endoplasmic_reticulum",
        "respiratory_complex_i-iv",
        "respiratory_complex_v",
        "nucleotide_metabolism",
    ]

format_volcano = pl.from_pandas(long_volcano_df).with_columns(
    pl.concat_str(pl.col("condition"), pl.lit(" vs. "),pl.col("control_condition")).alias("comparison")
).drop("p_value", "name", "condition", "control_condition", "-log10_pval_adj").rename({"-log10_pval":"neg_log10_pval"}).pivot(
    index=index_cols,
    on="comparison"
)

metrics = ["log2_FC", "Regulation", "neg_log10_pval"]
metric_pattern = f"^({'|'.join(metrics)})_(.*)"

def flip_column_name(col_name):
    # 'log2_FC_GroupA vs. GroupB' -> 'GroupA vs. GroupB_log2_FC'
    # for readability
    match = re.match(metric_pattern, col_name)
    if match:
        metric, comparison = match.groups()
        return f"{comparison}_{metric}"
    return col_name

# apply the rename
format_volcano = format_volcano.rename({
    col: flip_column_name(col) 
    for col in format_volcano.columns 
    if any(col.startswith(m) for m in metrics)
})

sorted_column_names = sorted(format_volcano.columns)
format_volcano = format_volcano.select(sorted_column_names)

df = percent_controls.select(id_columns + ["percent_control", "clean_channel_name"]).pivot(
    index=id_columns, values="percent_control", on="clean_channel_name"
)

# filter out contaminants and keratins
df = df.filter(
    pl.col("uniprot").str.contains("contaminant").not_(),
    pl.col("description").str.contains("Keratin").not_(),
)

cache = create_entry_cache(df)

formatted_df = df.with_columns(
    pl.col("uniprot")
    .map_elements(lambda x: get_function(x, cache), return_dtype=pl.String)
    .alias("uniprot_function")
).select(pl.col(["protein", "uniprot", "description"]), "uniprot_function", cs.contains("D"))

formatted_df = formatted_df.join(other = format_volcano, on = ["protein", "description"], how = "left")

type_filter = "standard"
if CURATED_FILTERS:
    type_filter = "curated"
formatted_df.write_excel(
    workbook=data_dir / f"mouse_wp_exhaustion_{type_filter}_filters.xlsx",
    table_style=f"Table Style Light 8",
    column_widths=150,
    freeze_panes=(1,1),
)

### Mouse human comparison

In [ ]:
# format homology data
homology_df = (
    pl.read_csv(data_dir / "jackson_homology_db.txt", separator="\t")
    .select(["DB Class Key", "Common Organism Name", "Symbol"])
    .group_by(["DB Class Key", "Common Organism Name"])
    .agg(pl.all())
    .pivot(index="DB Class Key", values="Symbol", on="Common Organism Name")
    .explode("human")
    .explode("mouse, laboratory")
)

mouse_volcano_df = (
    pl.read_csv(data_dir / "long_volcano_df.csv")
    .rename({"protein": "mouse, laboratory"})
    .join(other=homology_df, on="mouse, laboratory", how="left")
    .drop("name")
    .with_columns(pl.lit("mouse").alias("dataset"))
)

# format human volcano data
volcano_id_columns = ["uniprot", "protein", "description"]
functional_lists = [
    "mitochondrial",
    "peroxisome",
    "redox_related",
    "cell_cycle",
    "nucleotide_metabolism",
    "endoplasmic_reticulum",
]

human_volcano_df = (
    pl.from_pandas(
        # using pandas bc it has skip rows argument
        pd.read_excel(
            io="/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Exhaustion manuscript/12_Supplementary_data/Data S2_HS.xlsx",
            sheet_name="S2-1 Unenriched proteomics",
            skiprows=2,
        )
    )
    .select(~cs.contains("_rep"))
    .drop(["protein.1", "description.1"])
    .unpivot(index=volcano_id_columns + functional_lists)
    .with_columns(
        pl.col("variable").str.split("_Exhaustion WP - ").list.get(0).alias("field"),
        pl.col("variable")
        .str.split("_Exhaustion WP - ")
        .list.get(1)
        .str.split(" (")
        .list.get(0)
        .alias("comparison"),
    )
    # filtering out comparisons to D8Q for now
    .filter(~pl.col("comparison").str.starts_with("D8Q"))
    .with_columns(
        # quiescent -> acute
        pl.col("comparison")
        .str.split(" vs. ")
        .list.get(1)
        .str.replace("Q", "A")
        .alias("condition"),
        pl.col("comparison")
        .str.split(" vs. ")
        .list.get(0)
        .str.replace("Q", "A")
        .alias("control_condition"),
    )
    .drop(["variable", "comparison"])
    .pivot(
        on="field",
        values="value",
        index=volcano_id_columns
        + functional_lists
        + ["condition", "control_condition"],
    )
    .filter(pl.col("condition").str.contains("8"))
    .rename({"protein": "human"})
    .join(other=homology_df, on="human", how="left")
    .with_columns(
        pl.lit(False).alias("respiratory_complex_i-iv"),
        pl.lit(False).alias("respiratory_complex_v"),
        pl.lit("human").alias("dataset"),
    )
)

combined_volcano_data = pl.concat(
    items=[mouse_volcano_df.select(human_volcano_df.columns), human_volcano_df],
    how="vertical_relaxed",
)

combined_volcano_data.write_csv(data_dir / "combined_long_volcano_data.csv")

# Human TIL data

In [ ]:
data_dir = Path("/Users/henrysanford/dev/test_data/texh_revisions/new_human_til/output")

wp_channel_ratio_df = read_channel_ratios(
    data_dir=data_dir,
    donors={"HK3-81": "d1", "HK3-86": "d2"},
    folder_suffix="/03_combined_files",
    curated_filter_expression=(pl.col("protein").is_in(TRUE_POSITIVE_LIST))
    & CURATED_FILTERS,
    allow_one_rep=True,
    field_order = ["condition","donor","technical_replicate","dr"]
).filter(
    ~pl.col("condition").eq("Empty")
).with_columns(
    pl.col("condition").str.replace_all("-", " "),
    pl.col("clean_channel_name").str.replace_all("-", " ")
)

## Filter out hemoglobin proteins

In [ ]:
# display filtered out proteins
display(wp_channel_ratio_df.filter(hemoglobin_check).select("protein", "description").unique())
print(len(wp_channel_ratio_df))
# drop them
wp_channel_ratio_df = wp_channel_ratio_df.filter(~hemoglobin_check)
print(len(wp_channel_ratio_df))

In [ ]:
til_palette = {
'Human D0 CD3' : "white",
 'Human D2 CD3' : "#A1A1A1",
 'Human D8A CD3' : "#603EA6",
 'Human D8C CD3' : "#E27753",
 'TIL 1' : "#ac6468",
 'TIL 2' : "#a74245",
 'TIL 3' : "#d39e9c",
 'TIL 4' : "#7a2e31", 
 'TIL 5' : "#e8c4c3"  }

normalization_dir = data_dir / "normalization"
os.makedirs(normalization_dir, exist_ok = True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 5), sharey=True, sharex=True)

channel_ratio_boxplot(wp_channel_ratio_df, ax1, "Before normalization", color_palette=til_palette)

channel_medians = wp_channel_ratio_df.group_by(["donor", "clean_channel_name"]).agg(
    pl.col("value").median().alias("channel_median")
)

overall_median = wp_channel_ratio_df.select(pl.col("value").median()).item()

df_normalized = (
    wp_channel_ratio_df
    .join(channel_medians, on=["donor", "clean_channel_name"])
    .with_columns(
        (pl.col("value") * overall_median / pl.col("channel_median")).alias("value")
    )
    .drop("channel_median")
)
channel_ratio_boxplot(df_normalized, ax2, "After median normalization", color_palette=til_palette)

plt.tight_layout()

plt.savefig(
    normalization_dir / "median_normalization_boxplot.pdf"
)
plt.show()


### PCA

In [ ]:
pca_dir = data_dir / "pca"
os.makedirs(pca_dir, exist_ok=True)
pca_input_df = (
    df_normalized.drop("peptide_number")
    .pivot(index=id_columns, values="value", on="clean_channel_name")
    .drop(["uniprot", "description"])
    #.drop(cs.contains("d2"))
    .to_pandas()
)


pca_plt, loadings_df, pca_df, percent_explained_df = get_pca_plot(
    df=pca_input_df,
    index_cols="protein",
    title_string="Whole proteome",
    out_dir=Path(pca_dir),
)

# pca_plt.write_html("/Users/henrysanford/dev/normalized_pca")
pca_plt.show()

In [ ]:
do_pca(loadings_df, gmt, pca_dir)

In [ ]:
pca_dir = data_dir / "pca" / "in_vivo"
os.makedirs(pca_dir, exist_ok=True)
pca_input_df = (
    df_normalized.drop("peptide_number")
    .pivot(index=id_columns, values="value", on="clean_channel_name")
    # only TIL samples
    .select(~cs.contains("Human"))
    .drop(["uniprot", "description"])
    #.drop(cs.contains("d2"))
    .to_pandas()
)


pca_plt, loadings_df, pca_df, percent_explained_df = get_pca_plot(
    df=pca_input_df,
    index_cols="protein",
    title_string="Whole proteome",
    out_dir=Path(pca_dir),
)

# pca_plt.write_html("/Users/henrysanford/dev/normalized_pca")
pca_plt.show()

In [ ]:
do_pca(loadings_df, gmt, pca_dir)

In [ ]:
bargraph_dir = Path(data_dir  / "bargraphs")
bargraph_dir.mkdir(exist_ok=True)

percent_controls = channel_ratio_to_percent_control(
    df_normalized,
    "Human D2 CD3",
    id_columns + ["donor", "peptide_number"]
)



percent_controls.write_csv(
    bargraph_dir / "long_percent_control.csv"
)

In [ ]:
volcano_dir = data_dir / "volcano_plots"
os.makedirs(volcano_dir, exist_ok=True)

til_palette = {
    "Human D0 CD3": "white",
    "Human D2 CD3": "#A1A1A1",
    "Human D8A CD3": "#603EA6",
    "Human D8C CD3": "#E27753",
    "TIL 1": "#ac6468",
    "TIL 2": "#a74245",
    "TIL 3": "#d39e9c",
    "TIL 4": "#7a2e31",
    "TIL 5": "#e8c4c3",
}

comparisons = {
    "Human D2 CD3": [
        "Human D8A CD3",
        "Human D8C CD3",
        "TIL 1",
        "TIL 2",
        "TIL 3",
        "TIL 4",
        "TIL 5",
        "TIL",  # will select all the TILs
    ],
    "Human D8A CD3": [
        "Human D8A CD3",
        "Human D8C CD3",
        "TIL 1",
        "TIL 2",
        "TIL 3",
        "TIL 4",
        "TIL 5",
        "TIL",  # will select all the TILs
    ],
}

volcano_input = df_normalized.pivot(
    on="clean_channel_name", values="value", index=["uniprot", "protein", "description"]
).to_pandas()
results = []
for control, conditions in comparisons.items():
    volcano_df, copy_df, long_df = get_volcano_plot_treatment_vs_control(
        conditions,
        control,
        volcano_input,
        "Mouse WP",
        volcano_dir,
    )
    results.append(long_df)

In [ ]:
protein_list_dir = Path(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Vinogradova Laboratory/Henry_data processing/01_data_analysis_folders/03_Exhaustion/exhaustion/exhaustion_notebook/01_wp_analysis/"
)


def annotate_functional_categories(volcano_df):
    """Annotate the volcano_data plot with functional categories. See 'protein_lists/README.md' for
    further documentation
    """
    fun_groups = pd.read_csv(
        protein_list_dir / "protein_lists/group_annotations_final.csv"
    )
    for i, row in fun_groups.iterrows():
        column_name = row["column_name"]
        file_name = row["file_name"]
        print(f"Processing {column_name}...")

        protein_table = pd.read_csv(protein_list_dir / file_name)
        functional_proteins = set(protein_table["uniprot"])

        volcano_df[column_name] = volcano_df.index.isin(functional_proteins)

    return volcano_df


long_volcano_df = annotate_functional_categories(
    pl.from_pandas(pd.concat(results))
    .filter(~pl.col("uniprot").str.contains("contaminant"))
    .to_pandas()
    .set_index("uniprot")
)

long_volcano_df.to_csv(data_dir / "long_volcano_df.csv")

format_volcano = pl.from_pandas(long_volcano_df).with_columns(
    pl.concat_str(pl.col("condition"), pl.lit(" vs. "),pl.col("control_condition")).alias("comparison")
).drop("p_value", "name", "condition", "control_condition", "-log10_pval_adj").rename({"-log10_pval":"neg_log10_pval"}).pivot(
    index=[
        "protein",
        "description",
        # keep all the functional category annotataions
        "mitochondrial",
        "peroxisome",
        "redox_related",
        "cell_cycle",
        "endoplasmic_reticulum",
        "respiratory_complex_i-iv",
        "respiratory_complex_v",
        "nucleotide_metabolism",
    ],
    on="comparison",
)

In [ ]:
df = percent_controls.select(id_columns + ["percent_control", "clean_channel_name"]).pivot(
    index=id_columns, values="percent_control", on="clean_channel_name"
)

# filter out contaminants and keratins
df = df.filter(
    pl.col("uniprot").str.contains("contaminant").not_(),
    pl.col("description").str.contains("Keratin").not_(),
)

cache = create_entry_cache(df)

formatted_df = df.with_columns(
    pl.col("uniprot")
    .map_elements(lambda x: get_function(x, cache), return_dtype=pl.String)
    .alias("uniprot_function")
).select(pl.col(["protein", "uniprot", "description"]), "uniprot_function", cs.contains("D"))

formatted_df = formatted_df.join(other = format_volcano, on = ["protein", "description"], how = "left")

type_filter = "standard"
if CURATED_FILTERS:
    type_filter = "curated"
formatted_df.write_excel(
    workbook=data_dir / f"human_til_wp_exhaustion_{type_filter}_filters.xlsx",
    table_style=f"Table Style Light 8",
    column_widths=150,
    freeze_panes=(1,1),
)